In [0]:
sql_server   = dbutils.secrets.get(scope="retailpulse-scope", key="sql-server-vault")
sql_database = dbutils.secrets.get(scope="retailpulse-scope", key="db-name")
sql_username = dbutils.secrets.get(scope="retailpulse-scope", key="retailpulse-sql-username")
sql_password = dbutils.secrets.get(scope="retailpulse-scope", key="retailpulse-sql-password")

print(f"Server:   {sql_server}")
print(f"Database: {sql_database}")
print(f"Username: {sql_username}")
print(f"Password: {sql_password}")

In [0]:
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope ="retailpulse-scope", key= "retailpulse-account-key")
STORAGE_ACCOUNT_NAME = "retailpulsestorage" 
CONTAINER_NAME       = "raw-data"


spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.blob.core.windows.net",
    STORAGE_ACCOUNT_KEY
)


In [0]:
mount_path = dbutils.fs.ls(f"wasbs://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.blob.core.windows.net")

display(mount_path)

In [0]:
path = mount_path[1]

In [0]:
jdbc_url = (
    f"jdbc:sqlserver://{sql_server}:1433;"
    f"database={sql_database};"
    f"encrypt=true;"
    f"trustServerCertificate=false;"
    f"hostNameInCertificate=*.database.windows.net;"
    f"loginTimeout=30"
)

connection_properties = {
    "user":     sql_username,
    "password": sql_password,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Load Silver layer
df_clean = spark.read.parquet(f"{path.path}/clean_transactions")

print(f"Silver rows loaded: {df_clean.count():,}")

In [0]:
from pyspark.sql.functions import col, when

dim_customers = df_clean \
    .select("Customer ID", "Country") \
    .distinct() \
    .withColumnRenamed("Customer ID", "customer_id") \
    .withColumnRenamed("Country",     "country") \
    .withColumn("is_guest", when(col("customer_id") == "GUEST", 1).otherwise(0))

print(f"Unique customers: {dim_customers.count():,}")
display(dim_customers.limit(5))

# Write to SQL
dim_customers.write.jdbc(
    url=jdbc_url,
    table="dim_customers",
    mode="overwrite",
    properties=connection_properties
)

print("dim_customers written to SQL")

In [0]:
dim_products = df_clean \
    .select("StockCode", "Description") \
    .distinct() \
    .withColumnRenamed("StockCode",   "stock_code") \
    .withColumnRenamed("Description", "description")
from pyspark.sql.functions import count

dim_products = df_clean \
    .groupBy("StockCode", "Description") \
    .count() \
    .withColumn("rn", 
        col("count") / col("count")  
    )

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("StockCode").orderBy(desc("count"))

dim_products = df_clean \
    .groupBy("StockCode", "Description") \
    .count() \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .select(
        col("StockCode").alias("stock_code"),
        col("Description").alias("description")
    )

print(f"Unique products: {dim_products.count():,}")
display(dim_products.limit(5))

dim_products.write.jdbc(
    url=jdbc_url,
    table="dim_products",
    mode="overwrite",
    properties=connection_properties
)

print(" dim_products written to SQL")

In [0]:
from pyspark.sql.functions import (
    col, to_date, year, quarter, month, date_format,
    weekofyear, dayofmonth, dayofweek, when
)

#extract distinct dates from transactions
dim_date = df_clean \
    .select(to_date(col("InvoiceDate")).alias("full_date")) \
    .distinct() \
    .withColumn("date_key",     (year("full_date") * 10000 + 
                                 month("full_date") * 100 + 
                                 dayofmonth("full_date")).cast("int")) \
    .withColumn("year",         year("full_date")) \
    .withColumn("quarter",      quarter("full_date")) \
    .withColumn("month",        month("full_date")) \
    .withColumn("month_name",   date_format("full_date", "MMMM")) \
    .withColumn("week",         weekofyear("full_date")) \
    .withColumn("day_of_month", dayofmonth("full_date")) \
    .withColumn("day_of_week",  dayofweek("full_date")) \
    .withColumn("day_name",     date_format("full_date", "EEEE")) \
    .withColumn("is_weekend",   when(dayofweek("full_date").isin(1, 7), 1).otherwise(0)) \
    .orderBy("full_date")

print(f"Unique dates: {dim_date.count():,}")
display(dim_date.limit(5))

#write to SQL
dim_date.write.jdbc(
    url=jdbc_url,
    table="dim_date",
    mode="overwrite",
    properties=connection_properties
)

print("dim_date written to SQL")

In [0]:
for table in ["dim_customers", "dim_products", "dim_date"]:
    df = spark.read.jdbc(
        url=jdbc_url,
        table=f"(SELECT COUNT(*) as cnt FROM {table}) t",
        properties=connection_properties
    )
    count = df.collect()[0]["cnt"]
    print(f"{table}: {count:,} rows in SQL")

In [0]:
from pyspark.sql.functions import count, col

dupes = df_clean \
    .groupBy("StockCode") \
    .count() \
    .filter(col("count") > 1)

print(f"Stock codes with multiple rows: {dupes.count():,}")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, count, col, upper, trim

df_products_clean = df_clean \
    .withColumn("StockCode", upper(trim(col("StockCode"))))

window = Window.partitionBy("StockCode").orderBy(desc("count"))

dim_products = df_products_clean \
    .groupBy("StockCode", "Description") \
    .agg(count("*").alias("count")) \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .select(
        col("StockCode").alias("stock_code"),
        col("Description").alias("description")
    )

# Verify
total    = dim_products.count()
distinct = dim_products.select("stock_code").distinct().count()
print(f"Total rows:     {total:,}")
print(f"Distinct codes: {distinct:,}")
print(f"Duplicates:     {total - distinct}")

display(dim_products.limit(5))

In [0]:
def execute_ddl(statement):
    conn = (spark._jvm.java.sql.DriverManager.getConnection(
        jdbc_url, sql_username, sql_password
    ))
    stmt = conn.createStatement()
    stmt.execute(statement)
    stmt.close()
    conn.close()

execute_ddl("TRUNCATE TABLE dim_products")

dim_products.write \
    .format("jdbc") \
    .option("url",       jdbc_url) \
    .option("dbtable",   "dim_products") \
    .option("user",      sql_username) \
    .option("password",  sql_password) \
    .option("driver",    "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("batchsize", "5000") \
    .mode("append") \
    .save()

print(f"dim_products rewritten: {dim_products.count():,} rows")